<a href="https://colab.research.google.com/github/NataliaFarieta/nataliafarieta/blob/main/CuboDatos_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Taller ETL – Cubo SECOP
## Notebook 1 de 4: `CuboDatos.ipynb`
### Creación del modelo dimensional (cubo de datos) en Hive

**Universidad Central — Maestría en Analítica de Datos**
**Asignatura:** Big Data / Procesamiento Distribuido
**Actividad:** T2 – Taller ETL – Cubo SECOP
**Tipo de entregable:** Notebook JupyterLab (`*.ipynb`), PySpark + Hive

---

Este notebook implementa la **estructura** del cubo de datos de contratación
pública (SECOP II) en Hive, siguiendo fielmente el modelo definido en
`CuboContratosPostgreSQL.drawio`. No carga datos reales todavía — esa tarea
corresponde a `Cargue.ipynb`, una vez que `Extraccion.ipynb` y
`Transformacion.ipynb` hayan producido las capas BRONCE y PLATA del
*data lake*. Aquí solo se crea el esqueleto de tablas (hecho + dimensiones)
para que las capas posteriores tengan dónde escribir.


## 1. Introducción

El *Big Data* no solo implica volumen de información, sino la necesidad de
organizarla de forma que soporte análisis eficiente. En el contexto de la
contratación pública colombiana, el **SECOP II** (Sistema Electrónico de
Contratación Pública) expone, a través de una API abierta, miles de
registros de contratos estatales con decenas de atributos cada uno: valores,
fechas, entidades, proveedores, modalidades de contratación, ubicación
geográfica, etc.

Para explotar esta información con fines analíticos —por ejemplo, para
responder preguntas como "¿cuánto se contrató por departamento el año
pasado?" o "¿qué modalidad de contratación concentra más valor?"— es
necesario transformar el flujo de datos crudo en un **modelo dimensional**:
una tabla de **hechos** (los contratos, con sus medidas monetarias) rodeada
de **dimensiones** (entidad, proveedor, geografía, sector, etc.) que
permiten "cortar" (*slice*) y "agrupar" (*dice*) la información desde
distintos ángulos.

Este notebook construye ese esqueleto usando **PySpark** con soporte para
**Hive**, sobre una arquitectura de procesamiento distribuido compuesta por
Hadoop (HDFS como *data lake*) y Spark (motor de cómputo distribuido en
memoria), tal como se describe en Chambers y Zaharia (2018) y en la
documentación oficial de Apache Spark (Apache Software Foundation, 2025).


## 2. Objetivos

### Objetivo general

Implementar en Hive, sobre la arquitectura de procesamiento distribuido
Hadoop + Spark, la estructura del cubo de datos de contratación pública
SECOP, replicando fielmente el modelo dimensional definido en
`CuboContratosPostgreSQL.drawio`.

### Objetivos específicos

1. Crear una sesión de Spark con soporte de catálogo Hive habilitado.
2. Crear la base de datos Hive que alojará el cubo de datos.
3. Traducir el modelo `CuboContratosPostgreSQL.drawio` (tabla de hechos +
   9 dimensiones) a sentencias `CREATE TABLE` de Hive, respetando nombres
   de campos, tipos de dato y relaciones (PK/FK).
4. Verificar mediante consultas de catálogo (`SHOW DATABASES`,
   `SHOW TABLES`, `DESCRIBE`) que la estructura quedó correctamente creada.
5. Documentar el grano del hecho, las medidas y las relaciones del modelo
   para que sirvan de referencia a los notebooks siguientes.


## 3. Modelo dimensional — fuente de verdad: `CuboContratosPostgreSQL.drawio`

### 3.1 Tabla de hechos: `hecho_contratos`

**Grano del hecho:** una fila representa **un contrato** publicado en
SECOP II, identificado de forma única por `id_contrato`.

**Qué mide:** el hecho concentra las medidas monetarias asociadas al ciclo
de vida de un contrato (valor pactado, pagado, facturado, amortizado,
pendiente, etc.), junto con sus fechas clave y las llaves foráneas hacia
cada dimensión.

| Elemento | Detalle |
|---|---|
| PK | `id_contrato` |
| FK → Geografía | `geografia_id_geografia` |
| FK → Sector | `sector_id_sector` |
| FK → Proveedor | `proveedor_codigo_proveedor` |
| FK → Entidad | `entidad_codigo_entidad` |
| FK → Modalidad de contratación | `modalidad_id_modalidad_de_contratacion` |
| FK → Tipo de contrato | `contrato_id_tipo_de_contrato` |
| FK → Producto | `producto_codigo_de_producto` |
| FK → Ordenador de gasto | `ordenador_numero_de_documento_ordenador_del_gasto` |
| FK → Supervisor | `supervisor_numero_de_documento_supervisor` |
| Medidas | `valor_del_contrato`, `valor_pagado`, `valor_facturado`, `valor_pendiente_de_pago`, `valor_amortizado`, `valor_pendiente_de_amortizacion`, `valor_pendiente_de_ejecucion` |

**Análisis que permite:** agregación de valor contratado/pagado/pendiente
por cualquier combinación de dimensiones (tiempo, geografía, entidad,
proveedor, modalidad, sector, producto), seguimiento de ejecución
presupuestal y trazabilidad de responsables (ordenador de gasto,
supervisor, representante legal).

### 3.2 Dimensiones

| Dimensión | Tabla Hive | PK | Atributos principales |
|---|---|---|---|
| Geografía | `dim_geografia` | `id_geografia_dane` | `departamento`, `municipio` |
| Sector | `dim_sector` | `id_sector` | `sector` |
| Proveedor | `dim_proveedor` | `codigo_proveedor` | `tipodocproveedor`, `documento_proveedor`, `proveedor_adjudicado`, `es_grupo`, `es_pyme` |
| Entidad | `dim_entidad` | `codigo_entidad` | `nit_entidad`, `nombre_entidad`, `orden`, `rama`, `entidad_centralizada` |
| Modalidad de contratación | `dim_modalidad_contratacion` | `id_modalidad_de_contratacion` | `modalidad_de_contratacion`, `justificacion_modalidad_de` |
| Tipo de contrato | `dim_tipo_contrato` | `id_tipo_de_contrato` | `tipo_de_contrato` |
| Producto | `dim_producto` | `codigo_de_producto` | `nombre_de_producto`, `codigo_de_clase`, `nombre_de_clase`, `codigo_de_familia`, `nombre_de_familia`, `codigo_de_segmento`, `nombre_de_segmento` |
| Ordenador de gasto | `dim_ordenador_gasto` | `numero_de_documento_ordenador_del_gasto` | `tipo_de_documento_ordenador_del_gasto`, `nombre_ordenador_del_gasto` |
| Supervisor | `dim_supervisor` | `numero_de_documento_supervisor` | `tipo_de_documento_supervisor`, `nombre_supervisor` |

### 3.3 Mapeo Draw.io → Hive (correspondencia de elementos)

| Elemento draw.io | Tabla Hive | Tipo de dato Hive | PK/FK |
|---|---|---|---|
| `Hecho (Contratos)` | `hecho_contratos` | — (tabla completa) | contiene todas las FK |
| `id_contrato` | `hecho_contratos.id_contrato` | STRING | PK |
| `valor_del_contrato` | `hecho_contratos.valor_del_contrato` | DOUBLE | medida |
| `Dimension (Geografia)` | `dim_geografia` | — | — |
| `id_geografia_dane` | `dim_geografia.id_geografia_dane` | STRING | PK |
| `Dimensión (Sector)` | `dim_sector` | — | — |
| `id_sector` | `dim_sector.id_sector` | STRING | PK |
| `Dimension (Proveedor)` | `dim_proveedor` | — | — |
| `codigo_proveedor` | `dim_proveedor.codigo_proveedor` | STRING | PK |
| `Dimensión (Entidad)` | `dim_entidad` | — | — |
| `codigo_entidad` | `dim_entidad.codigo_entidad` | STRING | PK |
| `Dimensión (Modalidad de contratación)` | `dim_modalidad_contratacion` | — | — |
| `id_modalidad_de_contratacion` | `dim_modalidad_contratacion.id_modalidad_de_contratacion` | STRING | PK |
| `Dimensión (Tipo de contrato)` | `dim_tipo_contrato` | — | — |
| `id_tipo_de_contrato` | `dim_tipo_contrato.id_tipo_de_contrato` | STRING | PK |
| `Dimensión (Producto)` | `dim_producto` | — | — |
| `codigo_de_producto` | `dim_producto.codigo_de_producto` | STRING | PK |
| `Dimensión (Ordenador de gasto)` | `dim_ordenador_gasto` | — | — |
| `número_de_documento_ordenador_del_gasto` | `dim_ordenador_gasto.numero_de_documento_ordenador_del_gasto` | STRING | PK |
| `Dimensión (Supervisor)` | `dim_supervisor` | — | — |
| `número_de_documento_supervisor` | `dim_supervisor.numero_de_documento_supervisor` | STRING | PK |

> **Nota de fidelidad al modelo.** El archivo `.drawio` no especifica tipos
> de dato explícitos (solo nombres de campo). Se documenta aquí la decisión
> de tipado: identificadores y códigos como `STRING` (evita pérdida de
> ceros a la izquierda, p. ej. NITs o códigos DANE), valores monetarios
> como `DOUBLE`, y fechas como `DATE`. Esta decisión se mantiene de forma
> consistente en `Transformacion.ipynb` y `Cargue.ipynb` para no introducir
> contradicciones entre notebooks.

> **Nota sobre `dim_producto` y `dim_geografia`.** El endpoint público de
> SECOP II usado en `Extraccion.ipynb`
> (`datos.gov.co/resource/jbjy-vk9h.json`) no expone directamente un código
> DANE de geografía ni un código UNSPSC ya segmentado en
> familia/clase/segmento; expone `departamento`, `ciudad`,
> `codigo_de_categoria_principal` (UNSPSC) y campos afines. En
> `Transformacion.ipynb` se documenta explícitamente cómo se deriva cada
> dimensión a partir de los campos disponibles en la fuente, dejando
> constancia de cualquier atributo que quede vacío por no estar disponible
> en el origen — nunca se inventan valores.


## 4. Arquitectura general del pipeline

```
SECOP API  ->  EXTRACCIÓN  ->  DATA LAKE (BRONCE)  ->  TRANSFORMACIÓN (PySpark)
   ->  DATA LAKE (PLATA)  ->  NORMALIZACIÓN  ->  HECHOS Y DIMENSIONES
   ->  DATA LAKE (ORO, Parquet)  ->  HIVE  ->  CUBO DE DATOS
   ->  CONSULTAS SQL  ->  ANÁLISIS
```

- **BRONCE**: datos crudos tal como llegan de la API, sin transformar.
- **PLATA**: datos limpios, tipados y normalizados en el modelo dimensional.
- **ORO**: datos finales en formato columnar (Parquet), cargados en Hive,
  listos para consulta analítica.

Este notebook (`CuboDatos.ipynb`) opera sobre la capa **estructural** del
cubo (metadatos Hive), previa a la existencia de cualquier dato en BRONCE,
PLATA u ORO.


## 5. Creación de la sesión de Spark con soporte Hive

Se habilita `enableHiveSupport()` para que el catálogo de Spark SQL use
Hive Metastore, permitiendo que las tablas creadas aquí sean visibles y
consultables desde los demás notebooks del taller (`Extraccion.ipynb`,
`Transformacion.ipynb`, `Cargue.ipynb`), todos ejecutados sobre el mismo
clúster contenerizado (Hadoop + Spark + JupyterLab on Docker).


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("CuboDatos_SECOP") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.catalogImplementation", "hive") \
    .enableHiveSupport() \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("SparkSession creada correctamente.")
print("Version de Spark:", spark.version)


## 6. Creación de la base de datos del cubo

Se crea (si no existe) la base de datos `secop_cubo`, que agrupará la
tabla de hechos y las nueve dimensiones definidas en el modelo.


In [ ]:
DB_NAME = "secop_cubo"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_NAME}")
spark.catalog.setCurrentDatabase(DB_NAME)

print(f"Base de datos activa: {spark.catalog.currentDatabase()}")


## 7. Creación de las tablas de dimensiones

Cada dimensión se crea como tabla gestionada por Hive (`CREATE TABLE`,
formato Parquet), respetando el nombre y tipo de cada atributo tal como
aparece en `CuboContratosPostgreSQL.drawio`. Se usa `IF NOT EXISTS` para
que el notebook sea reejecutable sin fallar si ya existían las tablas.


In [ ]:
ddl_dimensiones = {

"dim_geografia": """
CREATE TABLE IF NOT EXISTS dim_geografia (
    id_geografia_dane STRING,
    departamento       STRING,
    municipio          STRING
)
USING PARQUET
""",

"dim_sector": """
CREATE TABLE IF NOT EXISTS dim_sector (
    id_sector STRING,
    sector    STRING
)
USING PARQUET
""",

"dim_proveedor": """
CREATE TABLE IF NOT EXISTS dim_proveedor (
    codigo_proveedor     STRING,
    tipodocproveedor      STRING,
    documento_proveedor   STRING,
    proveedor_adjudicado  STRING,
    es_grupo              STRING,
    es_pyme               STRING
)
USING PARQUET
""",

"dim_entidad": """
CREATE TABLE IF NOT EXISTS dim_entidad (
    codigo_entidad        STRING,
    nit_entidad           STRING,
    nombre_entidad        STRING,
    orden                 STRING,
    rama                  STRING,
    entidad_centralizada  STRING
)
USING PARQUET
""",

"dim_modalidad_contratacion": """
CREATE TABLE IF NOT EXISTS dim_modalidad_contratacion (
    id_modalidad_de_contratacion STRING,
    modalidad_de_contratacion    STRING,
    justificacion_modalidad_de   STRING
)
USING PARQUET
""",

"dim_tipo_contrato": """
CREATE TABLE IF NOT EXISTS dim_tipo_contrato (
    id_tipo_de_contrato STRING,
    tipo_de_contrato    STRING
)
USING PARQUET
""",

"dim_producto": """
CREATE TABLE IF NOT EXISTS dim_producto (
    codigo_de_producto  STRING,
    nombre_de_producto  STRING,
    codigo_de_clase     STRING,
    nombre_de_clase     STRING,
    codigo_de_familia   STRING,
    nombre_de_familia   STRING,
    codigo_de_segmento  STRING,
    nombre_de_segmento  STRING
)
USING PARQUET
""",

"dim_ordenador_gasto": """
CREATE TABLE IF NOT EXISTS dim_ordenador_gasto (
    numero_de_documento_ordenador_del_gasto STRING,
    tipo_de_documento_ordenador_del_gasto   STRING,
    nombre_ordenador_del_gasto              STRING
)
USING PARQUET
""",

"dim_supervisor": """
CREATE TABLE IF NOT EXISTS dim_supervisor (
    numero_de_documento_supervisor STRING,
    tipo_de_documento_supervisor   STRING,
    nombre_supervisor              STRING
)
USING PARQUET
""",

}

for nombre_tabla, ddl in ddl_dimensiones.items():
    spark.sql(ddl)
    print(f"OK -> {nombre_tabla}")

print("\nLas 9 tablas de dimensiones fueron creadas (o ya existian).")


## 8. Creación de la tabla de hechos

La tabla de hechos `hecho_contratos` incluye la PK, todas las FK hacia las
nueve dimensiones y las medidas monetarias, junto con las fechas y campos
descriptivos que trae el modelo original.


In [ ]:
ddl_hecho = """
CREATE TABLE IF NOT EXISTS hecho_contratos (
    -- Clave primaria y descriptores del contrato
    id_contrato                     STRING,
    referencia_del_contrato         STRING,
    estado_contrato                 STRING,
    descripcion_del_proceso         STRING,
    proceso_de_compra               STRING,
    objeto_del_contrato             STRING,

    -- Llaves foraneas hacia dimensiones
    geografia_id_geografia                             STRING,
    sector_id_sector                                   STRING,
    proveedor_codigo_proveedor                         STRING,
    entidad_codigo_entidad                             STRING,
    modalidad_id_modalidad_de_contratacion             STRING,
    contrato_id_tipo_de_contrato                       STRING,
    producto_codigo_de_producto                        STRING,
    ordenador_numero_de_documento_ordenador_del_gasto  STRING,
    supervisor_numero_de_documento_supervisor          STRING,

    -- Medidas (valores monetarios)
    valor_del_contrato               DOUBLE,
    valor_pagado                     DOUBLE,
    valor_facturado                  DOUBLE,
    valor_pendiente_de_pago          DOUBLE,
    valor_amortizado                 DOUBLE,
    valor_pendiente_de_amortizacion  DOUBLE,
    valor_pendiente_de_ejecucion     DOUBLE,

    -- Fuentes de financiacion
    presupuesto_general_de_la_nacion_pgn                             DOUBLE,
    sistema_general_de_participaciones                               DOUBLE,
    sistema_general_de_regalias                                      DOUBLE,
    recursos_propios_alcaldias_gobernaciones_y_resguardos_indigenas  DOUBLE,
    recursos_de_credito                                              DOUBLE,
    recursos_propios                                                 DOUBLE,

    -- Fechas
    fecha_de_firma               DATE,
    fecha_de_inicio_del_contrato DATE,
    fecha_de_fin_del_contrato    DATE,
    fecha_inicio_liquidacion     DATE,
    fecha_fin_liquidacion        DATE,
    ultima_actualizacion         DATE,

    -- Atributos descriptivos adicionales del hecho
    duracion_del_contrato                       STRING,
    dias_adicionados                            INT,
    obligacion_ambiental                        STRING,
    puntos_del_acuerdo                          STRING,
    pilares_del_acuerdo                         STRING,
    es_postconflicto                            STRING,
    nombre_representante_legal                  STRING,
    identificacion_representante_legal          STRING,
    tipo_de_identificacion_representante_legal  STRING
)
USING PARQUET
"""

spark.sql(ddl_hecho)
print("Tabla de hechos 'hecho_contratos' creada (o ya existia).")


## 9. Validación de la estructura creada

Se verifica, mediante consultas de catálogo, que la base de datos y las
10 tablas (1 hecho + 9 dimensiones) quedaron correctamente registradas en
el Metastore de Hive.


In [ ]:
spark.sql("SHOW DATABASES").show(truncate=False)


In [ ]:
spark.sql(f"SHOW TABLES IN {DB_NAME}").show(n=20, truncate=False)


In [ ]:
tablas = [r["tableName"] for r in spark.sql(f"SHOW TABLES IN {DB_NAME}").collect()]
esperadas = {
    "hecho_contratos", "dim_geografia", "dim_sector", "dim_proveedor",
    "dim_entidad", "dim_modalidad_contratacion", "dim_tipo_contrato",
    "dim_producto", "dim_ordenador_gasto", "dim_supervisor",
}
faltantes = esperadas - set(tablas)

print(f"Tablas encontradas ({len(tablas)}):", sorted(tablas))
if faltantes:
    print("ADVERTENCIA - faltan tablas:", faltantes)
else:
    print("VERIFICADO: las 10 tablas del cubo (1 hecho + 9 dimensiones) existen.")


In [ ]:
print("Esquema de la tabla de hechos:")
spark.sql("DESCRIBE hecho_contratos").show(n=50, truncate=False)


In [ ]:
for t in ["dim_geografia", "dim_sector", "dim_proveedor", "dim_entidad",
          "dim_modalidad_contratacion", "dim_tipo_contrato", "dim_producto",
          "dim_ordenador_gasto", "dim_supervisor"]:
    print(f"\n--- Esquema de {t} ---")
    spark.sql(f"DESCRIBE {t}").show(truncate=False)


## 10. Análisis

La estructura creada respeta la granularidad definida en el modelo
original: **una fila de `hecho_contratos` equivale a un contrato SECOP**,
y cada dimensión aporta una perspectiva analítica independiente
(geográfica, sectorial, de proveedor, de entidad contratante, de
modalidad de contratación, de tipo de contrato, de producto/servicio
adquirido, y de responsables administrativos — ordenador de gasto y
supervisor). Esta separación entre hechos (medidas numéricas) y
dimensiones (atributos descriptivos) es la base del **modelo estrella**
que permite luego construir agregaciones eficientes mediante `JOIN` y
`GROUP BY` en Spark SQL, sin necesidad de recorrer todo el detalle
transaccional en cada consulta.

Al usar formato **Parquet** (columnar) desde la creación de las tablas, se
sientan las bases para que las consultas del cubo —desarrolladas en
`Cargue.ipynb`— lean solo las columnas necesarias por consulta, reduciendo
significativamente el I/O frente a un formato de fila como CSV o JSON
(Chambers y Zaharia, 2018).

Todavía no existen registros cargados: eso ocurre progresivamente en
`Extraccion.ipynb` (BRONCE), `Transformacion.ipynb` (PLATA) y
`Cargue.ipynb` (ORO + carga final en estas mismas tablas Hive).


## 11. Conclusiones

- Se creó exitosamente la base de datos `secop_cubo` y las 10 tablas del
  modelo dimensional (1 hecho + 9 dimensiones), con nombres, tipos y
  relaciones fieles al archivo `CuboContratosPostgreSQL.drawio`.
- El uso de `enableHiveSupport()` permite que esta estructura sea
  compartida y reutilizada por los notebooks `Extraccion.ipynb`,
  `Transformacion.ipynb` y `Cargue.ipynb`, evitando duplicar definiciones
  de esquema.
- Quedó documentado el grano del hecho, sus medidas, sus relaciones y las
  decisiones de tipado adoptadas, como referencia normativa para el resto
  del taller.
- Los puntos donde el modelo `.drawio` no coincide exactamente con los
  campos disponibles en la fuente SECOP (geografía, producto) quedan
  señalados explícitamente, para resolverse con evidencia real en
  `Transformacion.ipynb`, sin inventar valores.

## 12. Referencias (APA 7)

Apache Software Foundation. (2025, 20 de agosto). *Apache Spark*.
https://spark.apache.org/

Chambers, B., & Zaharia, M. (2018). *Spark: The definitive guide*. O'Reilly Media.

Databricks. (s.f.). *What is PySpark?* https://www.databricks.com/glossary/pyspark

Karambelkar, H. V. (2018). *Apache Hadoop 3 quick start guide*. Packt Publishing.
